<a href="https://colab.research.google.com/github/omarsamehabobaker619-bot/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarsamehabobaker619-bot/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

print(con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1
"""))

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. What one row means: one page's performance on one day, for one client
   (report_date + client_hash_id + content_hash_id).

2. Table(s) used: fact_content_daily_performance, filtered to month=2026-03
   for development (the final month is a sealed test month, never used to
   build label logic).

3. Time window: a single mid-panel month, 2026-03 — one row per page-day
   within that month.

4. What I'd predict/rank: a proxy score — how far a page's CTR falls below
   what's typical for pages with similar average position, computed from
   gsc_clicks / gsc_impressions vs. gsc_avg_position. This is a proxy, not
   an observed label, since there's no column that directly states
   "underperforming."

5. What I deliberately exclude: the AI-referral columns (sessions_ai,
   ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta,
   ai_other) and the channel-breakdown session columns (sessions_organic,
   sessions_direct, etc.) — these belong to a different lane (AI-referral
   opportunity) and aren't needed for a CTR/position-based score.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature (used to predict):
- avg_position — pre-decision signal, from gsc_avg_position
- log_impressions — pre-decision signal, from gsc_impressions
- engaged_sessions — pre-decision signal, from ga4_engaged_sessions
- total_engagement_sec — pre-decision signal, from ga4_total_engagement_sec
- scroll_events — pre-decision signal, from scroll_events

Label (what's being predicted):
- is_underperforming — derived from ctr (gsc_clicks / gsc_impressions),
  compared against the median CTR across pages that month

Context (identifies the row, not used as a model input):
- report_date, client_hash_id, content_hash_id — identify which page,
  which client, which day; needed to trace results back, never fed to the model
- month — the partition key, confirms which slice of data this is

Excluded (and why):
- total_clicks / gsc_clicks — excluded from features because it directly
  builds the label; including it is the leakage trap demonstrated in Section 3
- sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude,
  ai_meta, ai_other — excluded because they belong to a different lane
  (AI-referral opportunity), not CTR/position scoring
- sessions_organic, sessions_direct, sessions_referral, sessions_social,
  sessions_paid — excluded for the same reason, channel breakdown isn't
  needed for this lane's score
- client_has_gsc, client_has_ga4, ga4_data_available — excluded as model
  inputs, but used as filters (via gsc_data_available IS TRUE) to decide
  which rows are usable at all

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS distinct_keys
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────┐
│ total_rows │ distinct_keys │
│   int64    │     int64     │
├────────────┼───────────────┤
│    9841378 │       9841378 │
└────────────┴───────────────┘



In [ ]:
span_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")
print(span_check)

┌───────────┬───────────────┬─────────────┐
│ row_count │ earliest_date │ latest_date │
│   int64   │     date      │    date     │
├───────────┼───────────────┼─────────────┤
│   9841378 │ 2026-03-01    │ 2026-03-31  │
└───────────┴───────────────┴─────────────┘



In [ ]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")
print(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │
│   int64    │       int64        │
├────────────┼────────────────────┤
│    9841378 │            3611061 │
└────────────┴────────────────────┘



In [ ]:
feature_frame = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(gsc_avg_position) AS avg_position,
        LN(SUM(gsc_impressions) + 1) AS log_impressions,
        SUM(ga4_engaged_sessions) AS engaged_sessions,
        SUM(ga4_total_engagement_sec) AS total_engagement_sec,
        SUM(scroll_events) AS scroll_events,
        SUM(gsc_clicks) AS total_clicks,
        SUM(gsc_impressions) AS total_impressions
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
    ORDER BY content_hash_id
    LIMIT 5000
""").df()

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,avg_position,log_impressions,engaged_sessions,total_engagement_sec,scroll_events,total_clicks,total_impressions
0,content_000005d4ced12088,72.854861,4.465908,0.0,0.0,0.0,0.0,86.0
1,content_00007bd2985b77c3,5.269565,3.871201,0.0,0.0,0.0,0.0,47.0
2,content_0000cd28fbda69f3,4.251282,3.401197,0.0,0.0,0.0,0.0,29.0
3,content_0000d495bfbfb4a8,3.333333,2.772589,0.0,0.0,0.0,0.0,15.0
4,content_00014efc121d911d,4.964683,4.762174,NaN,NaN,NaN,1.0,116.0


1. avg_position — knowable at the decision moment because it's computed
   from that day's GSC data, before any click outcome is known.
2. log_impressions — knowable at the decision moment because it's computed
   from that day's GSC impressions.
3. engaged_sessions — knowable at the decision moment because it's computed
   from that day's GA4 sync.
4. total_engagement_sec — knowable at the decision moment because it's
   computed from that day's GA4 sync.
5. scroll_events — knowable at the decision moment because it's computed
   from that day's GA4 sync.

In [ ]:
feature_frame["ctr"] = feature_frame["total_clicks"] / feature_frame["total_impressions"]
median_ctr = feature_frame["ctr"].median()
feature_frame["is_underperforming"] = (feature_frame["ctr"] <= median_ctr).astype(int)

feature_frame[["content_hash_id", "ctr", "is_underperforming"]].head()

,content_hash_id,ctr,is_underperforming
0,content_000005d4ced12088,0.000000,1
1,content_00007bd2985b77c3,0.000000,1
2,content_0000cd28fbda69f3,0.000000,1
3,content_0000d495bfbfb4a8,0.000000,1
4,content_00014efc121d911d,0.008621,0


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score

honest_features = ["avg_position", "log_impressions", "engaged_sessions", "total_engagement_sec", "scroll_events"]
X_honest = feature_frame[honest_features].fillna(0)
y = feature_frame["is_underperforming"]

model_honest = LogisticRegression(max_iter=1000).fit(X_honest, y)
preds_honest = model_honest.predict(X_honest)

print("Honest precision:", precision_score(y, preds_honest))

Honest precision: 0.8661926308985133


In [ ]:
leaked_features = ["avg_position", "log_impressions", "engaged_sessions", "total_engagement_sec", "scroll_events", "total_clicks"]
X_leaked = feature_frame[leaked_features].fillna(0)

model_leaked = LogisticRegression(max_iter=1000).fit(X_leaked, y)
preds_leaked = model_leaked.predict(X_leaked)

print("Leaked precision (with total_clicks added):", precision_score(y, preds_leaked))

Leaked precision (with total_clicks added): 1.0


The trap: adding total_clicks pushed precision from 0.8662 to 1.0. That's not
a real improvement — total_clicks directly built the ctr/label column, so the
model was just seeing the answer restated as a feature. This is leakage, not
signal. I'm removing total_clicks and keeping the honest score (0.8662) as the
real, defensible result.

Grain confirmed: row count equals distinct (client, page, day) keys. Date span is
the full month, March 2026, as expected. Only ~36.7% of rows have gsc_data_available
= TRUE — meaning most page-days in this table don't have usable GSC data, which any
feature built from GSC columns must filter for.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named limitation: this table's GSC availability is inconsistent — only about
36.7% of page-days have gsc_data_available = TRUE in March 2026. This means the
feature frame and label are both built from a partial slice of pages that happen
to have usable GSC data that month, not all pages equally. A page with sparse or
missing GSC data isn't necessarily low-performing — it may just be a page GSC
hasn't reliably reported on. Any conclusions from this notebook should be read
as applying to "pages with available GSC data in March 2026," not the full page
population.

A second limitation: GA4-derived features (engaged_sessions, total_engagement_sec,
scroll_events) weren't filtered on ga4_data_available. Per FlyRank's own data guidance,
GA4 columns are zero-filled before a client's GA4 sync started — so some zeros in
these features may mean "GA4 wasn't active yet," not "genuinely zero engagement."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- [x] Five plain-words contract answers provided
- [x] Exactly three verification queries with visible outputs (grain, row count/date span, availability with IS TRUE)
- [x] Five-feature frame built, each with an "available when?" line
- [x] Deliberate-leak experiment shown (precision jumped to 1.0) and removed, keeping the honest score (0.8662)
- [x] One named limitation stated (partial GSC availability, ~36.7%)
- [ ] Notebook runs top to bottom with no errors (Restart session → Run all) — do this now before checking this box

In [ ]:
schema_df = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1
""").df()

import pandas as pd
pd.set_option('display.max_rows', None)
print(schema_df[['column_name', 'column_type']])

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a